In [2]:
!pip install pulp

In [4]:
import numpy as np
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import pandas as pd
import time
import os
from openpyxl import load_workbook
from datetime import datetime

# Parámetros globales
directorio_inventarios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\invdiarios_todasbebe"
archivo_asimetria = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\asimetria_bikes_todascom_+20%.xlsx"
archivo_promedios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\promedio_bikes_por_estacion_todas.xlsx"
archivo_capacidades = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Capacidades_elegidas20todascom.xlsx"
archivo_costos = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\distancias_estaciones_todascom.xlsx"

output_file = "Resultado_optimizaciontodasanalisisbebe.xlsx"
hoja = "Sheet1"
Parametro_tiempo = 120

def fill_na(dataframe, fill_value=0):
    return dataframe.fillna(fill_value)

# Leer archivos fijos
promedios_df = fill_na(pd.read_excel(archivo_promedios))
capacidades_df = fill_na(pd.read_excel(archivo_capacidades))
costos_df = fill_na(pd.read_excel(archivo_costos))
asimetria_df_completo = fill_na(pd.read_excel(archivo_asimetria))

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1

# Preparar iteración
archivos_inventario = sorted([f for f in os.listdir(directorio_inventarios) if f.endswith(".xlsx")])
columnas_asimetria = ['Asi-50', 'Asi-25', 'Asi', 'Asi+25', 'Asi+50']

for archivo_inv in archivos_inventario:
    inventarios_df = fill_na(pd.read_excel(os.path.join(directorio_inventarios, archivo_inv)))

    for columna_asim in columnas_asimetria:
        start_time = time.time()
        print(f"Ejecutando: {archivo_inv} con asimetría {columna_asim}")

        Inventario_inicio, Inventario_final, Estacion = [], [], []
        estaciones_flujo, valores_flujo = [], []
        limites_superiores, limites_inferiores = [], []

        b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).fillna(0).to_numpy()
        M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).fillna(1).to_numpy()

        # ✅ APLICA max(0, ASI) para la columna correspondiente
        asi_values = asimetria_df_completo.set_index('Estacion')[columna_asim].reindex(unique_stations).fillna(0).to_numpy()
        MAX = np.maximum(0, asi_values)

        # MATRIZ DE COSTOS
        C = np.zeros((I, I))
        for _, row in costos_df.iterrows():
            i = station_to_index[row['Estacion_Origen']]
            j = station_to_index[row['Estacion_Destino']]
            C[i, j] = row['Costo_Salida']

        # CREACIÓN DEL MODELO
        model = LpProblem(name="problema-rebalanceo-bicicletas", sense=LpMinimize)

        B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
        FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
        Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

        model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            model += B[i, 0] == inventario_inicial
            model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)

        for i in range(I):
            model += B[i, 1] <= M[i]
            model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
            for j in range(I):
                if i != j:
                    model += FS[i, j, 0] <= M[i] * Y[i, j, 0]
            ajuste_sup = (1 + (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            ajuste_inf = (1 - (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            model += B[i, 1] <= ajuste_sup
            model += B[i, 1] >= ajuste_inf
            limites_superiores.append(ajuste_sup)
            limites_inferiores.append(ajuste_inf)

        status = model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
        costo_total = model.objective.value()
        contador = 0
        for i in range(I):
            for j in range(I):
                if i != j:
                    flujo = FS[i, j, 0].varValue
                    if flujo > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(flujo)
                        contador += 1

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            Inventario_inicio.append(int(inventario_inicial))
            Inventario_final.append(int(B[i, 1].varValue))
            Estacion.append(unique_stations[i])

        elapsed_time = time.time() - start_time
        fecha_valor = inventarios_df['Fecha_Hora'].tolist()[0] if 'Fecha_Hora' in inventarios_df.columns else archivo_inv

        df_resultado = pd.DataFrame({
            "Fecha": [fecha_valor],
            "Costo_total": [costo_total],
            "Parametro_tiempo": [Parametro_tiempo],
            "Tiempo_de_Ejecucion": [elapsed_time],
            "Estaciones": [Estacion],
            "Inventario_inicial": [Inventario_inicio],
            "Asimetria": [columna_asim],
            "Inventario_final": [Inventario_final],
            "Cantidad Total de Flujos": [contador],
            "Estaciones_Flujos": [estaciones_flujo],
            "Cantidad_Flujos": [valores_flujo],
            "Limites": [[(float(limites_superiores[i]), float(limites_inferiores[i])) for i in range(I)]]
        })

        try:
            if os.path.exists(output_file):
                book = load_workbook(output_file)
                if hoja in book.sheetnames:
                    startrow = book[hoja].max_row
                else:
                    startrow = 0
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=startrow == 0, startrow=startrow)
            else:
                with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False)

        except Exception as e:
            print(f"Error al guardar resultados: {e}")

print("¡Todos los modelos han sido ejecutados y guardados correctamente!")


Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi-25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi+25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi+50
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi-25
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi+25
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi+50
¡Todos los modelos han sido ejecutados y guardados correctamente!


In [5]:
import numpy as np
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import pandas as pd
import time
import os
from openpyxl import load_workbook
from datetime import datetime

# Parámetros globales
directorio_inventarios = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\invdiarios_todasbebe"
archivo_asimetria = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\asimetria_bikes_todascom_+20%.xlsx"
archivo_promedios = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\promedio_bikes_por_estacion_todas.xlsx"
archivo_capacidades = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\Capacidades_elegidas20todascom.xlsx"
archivo_costos = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\distancias_estaciones_todascom.xlsx"

output_file = "Resultado_optimizaciontodasanalisisbebe.xlsx"
hoja = "Sheet1"
Parametro_tiempo = 120

def fill_na(dataframe, fill_value=0):
    return dataframe.fillna(fill_value)

# Leer archivos fijos
promedios_df = fill_na(pd.read_excel(archivo_promedios))
capacidades_df = fill_na(pd.read_excel(archivo_capacidades))
costos_df = fill_na(pd.read_excel(archivo_costos))
asimetria_df_completo = fill_na(pd.read_excel(archivo_asimetria))

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1

# Preparar iteración
archivos_inventario = sorted([f for f in os.listdir(directorio_inventarios) if f.endswith(".xlsx")])
columnas_asimetria = ['Asi-50', 'Asi-25', 'Asi', 'Asi+25', 'Asi+50']

for columna_asim in columnas_asimetria:
    asimetria_columna = asimetria_df_completo[['Estacion', columna_asim]].copy()
    for archivo_inv in archivos_inventario:
        inventarios_df = fill_na(pd.read_excel(os.path.join(directorio_inventarios, archivo_inv)))

        start_time = time.time()
        print(f"Ejecutando: {archivo_inv} con variación de asimetría {columna_asim}")

        Inventario_inicio, Inventario_final, Estacion = [], [], []
        estaciones_flujo, valores_flujo = [], []
        limites_superiores, limites_inferiores = [], []

        b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).fillna(0).to_numpy()
        M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).fillna(1).to_numpy()

        asi_values = asimetria_columna.set_index('Estacion')[columna_asim].reindex(unique_stations).fillna(0).to_numpy()
        MAX = np.maximum(0, asi_values)

        C = np.zeros((I, I))
        for _, row in costos_df.iterrows():
            i = station_to_index[row['Estacion_Origen']]
            j = station_to_index[row['Estacion_Destino']]
            C[i, j] = row['Costo_Salida']

        model = LpProblem(name="problema-rebalanceo-bicicletas", sense=LpMinimize)

        B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
        FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
        Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

        model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            model += B[i, 0] == inventario_inicial
            model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)

        for i in range(I):
            model += B[i, 1] <= M[i]
            model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
            for j in range(I):
                if i != j:
                    model += FS[i, j, 0] <= M[i] * Y[i, j, 0]
            ajuste_sup = (1 + (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            ajuste_inf = (1 - (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            model += B[i, 1] <= ajuste_sup
            model += B[i, 1] >= ajuste_inf
            limites_superiores.append(ajuste_sup)
            limites_inferiores.append(ajuste_inf)

        status = model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
        costo_total = model.objective.value()
        contador = 0
        for i in range(I):
            for j in range(I):
                if i != j:
                    flujo = FS[i, j, 0].varValue
                    if flujo > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(flujo)
                        contador += 1

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            Inventario_inicio.append(int(inventario_inicial))
            Inventario_final.append(int(B[i, 1].varValue))
            Estacion.append(unique_stations[i])

        elapsed_time = time.time() - start_time
        fecha_valor = inventarios_df['Fecha_Hora'].tolist()[0] if 'Fecha_Hora' in inventarios_df.columns else archivo_inv

        df_resultado = pd.DataFrame({
            "Fecha": [fecha_valor],
            "Costo_total": [costo_total],
            "Parametro_tiempo": [Parametro_tiempo],
            "Tiempo_de_Ejecucion": [elapsed_time],
            "Estaciones": [Estacion],
            "Inventario_inicial": [Inventario_inicio],
            "Asimetria": [asi_values.tolist()],
            "Inventario_final": [Inventario_final],
            "Cantidad Total de Flujos": [contador],
            "Estaciones_Flujos": [estaciones_flujo],
            "Cantidad_Flujos": [valores_flujo],
            "Limites": [[(float(limites_superiores[i]), float(limites_inferiores[i])) for i in range(I)]],
            "Variacion asi": [columna_asim]
        })

        try:
            if os.path.exists(output_file):
                book = load_workbook(output_file)
                if hoja in book.sheetnames:
                    startrow = book[hoja].max_row
                else:
                    startrow = 0
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=startrow == 0, startrow=startrow)
            else:
                with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False)
        except Exception as e:
            print(f"Error al guardar resultados: {e}")

print("¡Todos los modelos han sido ejecutados y guardados correctamente!")


Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi-25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi+25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi+50
¡Todos los modelos han sido ejecutados y guardados correctamente!


In [2]:
import numpy as np
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import pandas as pd
import time
import os
from openpyxl import load_workbook
from datetime import datetime

# Parámetros globales
directorio_inventarios = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\invdiarios_caso1"
archivo_asimetria = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\Asimetria_E20_variacion.xlsx"
archivo_promedios = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\promedio_bikes_por_estacion_20.xlsx"
archivo_capacidades = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\Capacidades_elegidas20.xlsx"
archivo_costos = r"C:\\Users\\ACER\\Desktop\\Tesis\\Tesis Pedro Palominos\\Código python\\Modelo_info_historica\\Modelo más actualizado\\analisis sensibilidad\\analisis aumentar\\distancias_estaciones_20.xlsx"

output_file = "Resultado_optimizacioncaso1MALfinal.xlsx"
hoja = "Sheet1"
Parametro_tiempo = 120

def fill_na(dataframe, fill_value=0):
    return dataframe.fillna(fill_value)

# Leer archivos fijos
promedios_df = fill_na(pd.read_excel(archivo_promedios))
capacidades_df = fill_na(pd.read_excel(archivo_capacidades))
costos_df = fill_na(pd.read_excel(archivo_costos))
asimetria_df_completo = fill_na(pd.read_excel(archivo_asimetria))

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1

# Preparar iteración
archivos_inventario = sorted([f for f in os.listdir(directorio_inventarios) if f.endswith(".xlsx")])
columnas_asimetria = ['Asi-50', 'Asi-25', 'Asi', 'Asi+25', 'Asi+50']

for archivo_inv in archivos_inventario:
    inventarios_df = fill_na(pd.read_excel(os.path.join(directorio_inventarios, archivo_inv)))

    for columna_asim in columnas_asimetria:
        asimetria_columna = asimetria_df_completo[['Estacion', columna_asim]].copy()

        start_time = time.time()
        print(f"Ejecutando: {archivo_inv} con variación de asimetría {columna_asim}")

        Inventario_inicio, Inventario_final, Estacion = [], [], []
        estaciones_flujo, valores_flujo = [], []
        limites_superiores, limites_inferiores = [], []

        b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).fillna(0).to_numpy()
        M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).fillna(1).to_numpy()

        asi_values_raw = asimetria_columna.set_index('Estacion')[columna_asim].reindex(unique_stations).fillna(0).to_numpy()
        asi_values = np.maximum(0, asi_values_raw)

        C = np.zeros((I, I))
        for _, row in costos_df.iterrows():
            i = station_to_index[row['Estacion_Origen']]
            j = station_to_index[row['Estacion_Destino']]
            C[i, j] = row['Costo_Salida']

        model = LpProblem(name="problema-rebalanceo-bicicletas", sense=LpMinimize)

        B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
        FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
        Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

        model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            model += B[i, 0] == inventario_inicial
            model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)

        for i in range(I):
            model += B[i, 1] <= M[i]
            model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
            for j in range(I):
                if i != j:
                    model += FS[i, j, 0] <= M[i] * Y[i, j, 0]
            ajuste_sup = (1 + (b[i] / M[i]) + (asi_values[i] / M[i])) * (M[i] / 2)
            ajuste_inf = (1 - (b[i] / M[i]) + (asi_values[i] / M[i])) * (M[i] / 2)
            model += B[i, 1] <= ajuste_sup
            model += B[i, 1] >= ajuste_inf
            limites_superiores.append(ajuste_sup)
            limites_inferiores.append(ajuste_inf)

        status = model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
        costo_total = model.objective.value()
        contador = 0
        for i in range(I):
            for j in range(I):
                if i != j:
                    flujo = FS[i, j, 0].varValue
                    if flujo > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(flujo)
                        contador += 1

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            Inventario_inicio.append(int(inventario_inicial))
            Inventario_final.append(int(B[i, 1].varValue))
            Estacion.append(unique_stations[i])

        elapsed_time = time.time() - start_time
        fecha_valor = inventarios_df['Fecha_Hora'].tolist()[0] if 'Fecha_Hora' in inventarios_df.columns else archivo_inv

        df_resultado = pd.DataFrame({
            "Fecha": [fecha_valor],
            "Costo_total": [costo_total],
            "Parametro_tiempo": [Parametro_tiempo],
            "Tiempo_de_Ejecucion": [elapsed_time],
            "Estaciones": [Estacion],
            "Inventario_inicial": [Inventario_inicio],
            "Asimetria": [asi_values.tolist()],
            "Inventario_final": [Inventario_final],
            "Cantidad Total de Flujos": [contador],
            "Estaciones_Flujos": [estaciones_flujo],
            "Cantidad_Flujos": [valores_flujo],
            "Limites": [[[float(limites_superiores[i]), float(limites_inferiores[i])] for i in range(I)]],
            "Variacion asi": [columna_asim]
        })

        try:
            if os.path.exists(output_file):
                book = load_workbook(output_file)
                if hoja in book.sheetnames:
                    startrow = book[hoja].max_row
                else:
                    startrow = 0
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=startrow == 0, startrow=startrow)
            else:
                with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False)
        except Exception as e:
            print(f"Error al guardar resultados: {e}")

print("¡Todos los modelos han sido ejecutados y guardados correctamente!")


Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi-25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi+25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con variación de asimetría Asi+50
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con variación de asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con variación de asimetría Asi-25
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con variación de asimetría Asi
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con variación de asimetría Asi+25
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con variación de asimetría Asi+50
Ejecutando: bikes_disponibles_2024-11-10_18-00.xlsx con variación de asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-10_18-00.xlsx con vari

In [ ]:
#"Capacidad": [M],